# Step 4 — 分類器訓練與評估
載入向量，依時間順序切割訓練/測試集，訓練 NB / KNN / SVM / DT / RF 並輸出混淆矩陣

Big Data & Business Analytics, National Taiwan University

In [23]:
# ── 可調整參數區 ──────────────────────────────────────
TEST_RATIO = 0.2   # 測試資料比例（後 20% 時間段）
# ──────────────────────────────────────────────────────

In [24]:
import pickle
import numpy as np
from sklearn.naive_bayes      import MultinomialNB
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.svm              import SVC
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import confusion_matrix, accuracy_score

# [METHOD] 目前方法：時序切割 80/20 | 可替換為：移動回測（step5）
# [METHOD] 目前方法：NB/KNN/SVM/DT/RF | 可替換為：投票集成、LLM 零樣本分類

In [25]:
# 載入 vectors.pkl
f = open('vectors.pkl', 'rb')
data = pickle.load(f)
f.close()

feature_list = data['feature_list']
X = np.array(data['X'])       # shape: (n_articles, n_features)
y = np.array(data['y'])       # shape: (n_articles,)

print(f'共 {len(X)} 篇文章，{len(feature_list)} 維特徵')
print(f'看漲：{sum(y==1)} 篇，看跌：{sum(y==-1)} 篇')

共 2691 篇文章，500 維特徵
看漲：1596 篇，看跌：1095 篇


In [26]:
# 依時間順序切割（articles 已按 post_time 排序）
# 不用隨機切割，保持時間先後
split_idx = int(len(X) * (1 - TEST_RATIO))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'訓練集：{len(X_train)} 篇，測試集：{len(X_test)} 篇')

訓練集：2152 篇，測試集：539 篇


In [27]:
# 評估函式（訓練 → 預測 → 印出混淆矩陣與準確率）
def evaluate(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    cm     = confusion_matrix(y_test, y_pred, labels=[1, -1])
    print(f'\n===== {name} =====')
    print(f'準確率：{acc*100:.1f}%')
    print('Confusion Matrix:')
    print(f'{"":16s} 預測漲  預測跌')
    print(f'{"真實漲":16s}  {cm[0][0]:5d}    {cm[0][1]:5d}')
    print(f'{"真實跌":16s}  {cm[1][0]:5d}    {cm[1][1]:5d}')
    return acc

In [28]:
# Naive Bayes（需要非負輸入，TF 向量天然滿足）
acc_nb = evaluate('Naive Bayes', MultinomialNB(), X_train, y_train, X_test, y_test)


===== Naive Bayes =====
準確率：59.7%
Confusion Matrix:
                 預測漲  預測跌
真實漲                 108      111
真實跌                 106      214


In [29]:
# KNN（k=5）
acc_knn = evaluate('KNN (k=5)', KNeighborsClassifier(n_neighbors=5), X_train, y_train, X_test, y_test)


===== KNN (k=5) =====
準確率：46.2%
Confusion Matrix:
                 預測漲  預測跌
真實漲                 171       48
真實跌                 242       78


In [30]:
# SVM（線性核）
acc_svm = evaluate('SVM', SVC(kernel='linear'), X_train, y_train, X_test, y_test)


===== SVM =====
準確率：55.3%
Confusion Matrix:
                 預測漲  預測跌
真實漲                 155       64
真實跌                 177      143


In [31]:
# Decision Tree
acc_dt = evaluate('Decision Tree', DecisionTreeClassifier(random_state=42), X_train, y_train, X_test, y_test)


===== Decision Tree =====
準確率：48.6%
Confusion Matrix:
                 預測漲  預測跌
真實漲                 131       88
真實跌                 189      131


In [32]:
# Random Forest
acc_rf = evaluate('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, X_test, y_test)


===== Random Forest =====
準確率：51.4%
Confusion Matrix:
                 預測漲  預測跌
真實漲                 151       68
真實跌                 194      126


In [34]:
# 準確率彙整
print('\n===== 準確率彙整 =====')
results = [
    ('Naive Bayes',   acc_nb),
    ('KNN (k=5)',     acc_knn),
    ('SVM',           acc_svm),
    ('Decision Tree', acc_dt),
    ('Random Forest', acc_rf),
]
for name, acc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f'  {name:20s}: {acc*100:.1f}%')


===== 準確率彙整 =====
  Naive Bayes         : 59.7%
  SVM                 : 55.3%
  Random Forest       : 51.4%
  Decision Tree       : 48.6%
  KNN (k=5)           : 46.2%
